In [ ]:
!pip install -q datasets transformers huggingface_hub accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [ ]:
# Import the libraries
import torch
torch.cuda.is_available()

True

In [ ]:
# Pre process the data
from datasets import load_dataset
import numpy as np

ds = load_dataset("AdityaAI9/Sentiment_analysis_finance_dataset")

print("Original dataset columns:", ds["train"].column_names)

label_column = [col for col in ds["train"].column_names if col != "text"][0]

train_split = ds["train"].class_encode_column(label_column)

num_classes = len(train_split.features[label_column].names)
print(f"Número de clases detectadas: {num_classes}")

if label_column != "label":
    train_split = train_split.rename_column(label_column, "label")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

Financial_Sentiment_data.csv:   0%|          | 0.00/4.51M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/31603 [00:00<?, ? examples/s]

Original dataset columns: ['text', 'sentiment']


Casting to class labels:   0%|          | 0/31603 [00:00<?, ? examples/s]

Número de clases detectadas: 3


In [ ]:
# Splitting the data set for scalability purposes
#small_train_dataset = ds["train"].shuffle(seed=42).select([i for i in list(range(5000))])
#small_test_dataset = ds["test"].shuffle(seed=42).select([i for i in list(range(500))])

# Modification v2.
# Combined slice of 5,500 samples from the available 'train' key
small_dataset = train_split.shuffle(seed=42).select(range(20000))

# Programmatically generate a distinct test split
split_dataset = small_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

In [ ]:
# Tokenizer
from transformers import AutoTokenizer
model_checkpoint = "FacebookAI/xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [ ]:
# Map method to the train and test splitting process
def preprocess_function(examples):
   safe_text = [str(text) if text is not None else "" for text in examples["text"]]
   return tokenizer(safe_text, truncation=True)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
# Speed up with the conversion of the trainning samples to PyTorch tensors
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
## TRAINNING THE MODEL
# Selection of the pretrained model
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("FacebookAI/xlm-roberta-base", num_labels=num_classes)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Evaluation Metrics
import numpy as np
import evaluate

def compute_metrics(eval_pred):
   load_accuracy = evaluate.load("accuracy")
   load_f1 = evaluate.load("f1")

   logits, labels = eval_pred
   predictions = np.argmax(logits, axis=-1)
   print(predictions)
   print(labels)

   accuracy = load_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
   f1 = load_f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
   return {"accuracy": accuracy, "f1": f1}


In [ ]:
# Login to HuggingFace
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import TrainingArguments, Trainer

repo_name = "SamuelCanedo/finetuning-sentiment-model-3000-samples"

training_args = TrainingArguments(
   output_dir=repo_name,
   learning_rate=1e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=5,
   weight_decay=0.02,
   eval_strategy="epoch",           # Added tracking strategy
   save_strategy="epoch",
   load_best_model_at_end=True,  # Keeps the best version, not just the last one
   metric_for_best_model="f1",
   push_to_hub=True,
)

trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   processing_class=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.027964,0.006123,0.998500,0.998487
2,0.015785,0.003268,0.998500,0.998487
3,0.006130,0.002301,0.999500,0.999486
4,0.003412,0.004011,0.998500,0.998501
5,0.001294,0.001634,0.999500,0.999529


[0 0 1 ... 2 0 0]
[0 0 1 ... 2 0 0]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[0 0 1 ... 2 0 0]
[0 0 1 ... 2 0 0]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

[0 0 1 ... 2 0 0]
[0 0 1 ... 2 0 0]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[0 0 1 ... 2 0 0]
[0 0 1 ... 2 0 0]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5625, training_loss=0.028613953570028147, metrics={'train_runtime': 1453.627, 'train_samples_per_second': 61.914, 'train_steps_per_second': 3.87, 'total_flos': 1843117713478944.0, 'train_loss': 0.028613953570028147, 'epoch': 5.0})

In [ ]:
trainer.evaluate()

[0 0 1 ... 2 0 0]
[0 0 1 ... 2 0 0]


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.001294,0.001634,5,0.999500,0.999529


{'eval_loss': 0.001633681240491569,
 'eval_accuracy': 0.9995,
 'eval_f1': 0.9995287556403308}

In [ ]:
# Testing
from transformers import pipeline
import numpy as np

# 1. Define your personal model path on the Hub
model_id = "SamuelCanedo/finetuning-sentiment-model-3000-samples"

print("Loading your custom model from Hugging Face Hub... (This may take a moment to download)")
# Using the pipeline utility automates tokenization and tensor matching completely
sentiment_pipeline = pipeline("text-classification", model=model_id)
print("Model loaded successfully!\n")

# 2. Provide realistic financial sentences to test its accuracy
test_phrases = [
    "Despite a severe 15% collapse in year-over-year smartphone shipments, Apple managed to beat quarterly revenue estimates due to unprecedented growth in its services sector.",
    "The regulatory committee announced it has officially dropped all antitrust investigations against the tech giant, effectively removing the dark cloud that has suppressed its market valuation for months.",
    "The company proudly boasted a record-breaking 50% reduction in net losses, though total revenue simultaneously plunged to an all-time low."
]

# 3. Define the label mapping from your training step
# (0 = Negative, 1 = Neutral, 2 = Positive)
label_map = {
    "LABEL_0": "NEGATIVE 📉",
    "LABEL_1": "NEUTRAL 📊",
    "LABEL_2": "POSITIVE 📈"
}

# 4. Run prediction and output the results neatly
print("--- MODEL PREDICTIONS ---")
results = sentiment_pipeline(test_phrases)

for phrase, result in zip(test_phrases, results):
    raw_label = result['label']
    readable_label = label_map.get(raw_label, raw_label)
    confidence_score = result['score'] * 100

    print(f"Text: \"{phrase}\"")
    print(f"Result: {readable_label} ({confidence_score:.2f}% Confidence)\n")

Loading your custom model from Hugging Face Hub... (This may take a moment to download)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully!

--- MODEL PREDICTIONS ---
Text: "Despite a severe 15% collapse in year-over-year smartphone shipments, Apple managed to beat quarterly revenue estimates due to unprecedented growth in its services sector."
Result: NEGATIVE 📉 (99.82% Confidence)

Text: "The regulatory committee announced it has officially dropped all antitrust investigations against the tech giant, effectively removing the dark cloud that has suppressed its market valuation for months."
Result: NEUTRAL 📊 (92.98% Confidence)

Text: "The company proudly boasted a record-breaking 50% reduction in net losses, though total revenue simultaneously plunged to an all-time low."
Result: POSITIVE 📈 (54.40% Confidence)



Chinese dataset

In [ ]:
# loading chinese dataset
from datasets import load_dataset

ds = load_dataset("JunyuLu/ToxiCN")